In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D13 — Eurostat — Unemployment Rates by Country of Birth
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

import hashlib
import json
import re

import pandas as pd

DOCUMENT_ID = "D13"
DOCUMENT_NAME = "Eurostat — Unemployment rates by country of birth"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "Deterministic complete XLSX-to-coordinate-aware Markdown "
    "conversion preserving every non-empty cell with worksheet, "
    "cell coordinate, source type and value"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".xlsx"
EXPECTED_SOURCE_SHA256 = "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a"

SUMMARY_SHEET = "Summary"

EXPECTED_SHEETS = [
    "Summary"
] + [
    f"Sheet {number}"
    for number in range(1, 16)
]

SELECTED_SHEETS = [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5"
]

SELECTED_GEOGRAPHIES = [
    "European Union - 27 countries (from 2020)",
    "Belgium",
    "Germany",
    "Spain",
    "Portugal"
]

SELECTED_YEARS = [
    2020,
    2022,
    2024
]

EXPECTED_RECORD_COUNT = (
    len(SELECTED_SHEETS)
    * len(SELECTED_GEOGRAPHIES)
    * len(SELECTED_YEARS)
)

EXPECTED_YEAR_COUNTS = {
    str(year):
        len(SELECTED_SHEETS)
        * len(SELECTED_GEOGRAPHIES)
    for year in SELECTED_YEARS
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

EXPECTED_CATEGORY = "Labour market time series"
EXPECTED_TOPIC = "Unemployment rate by country of birth"
EXPECTED_UNIT = "percent"

EXPECTED_CATEGORY_COUNTS = {
    EXPECTED_CATEGORY: EXPECTED_RECORD_COUNT
}

ALLOWED_CATEGORIES = {EXPECTED_CATEGORY}

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

OUTPUT_DIR = Path("outputs_D13_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTICS_PATH = OUTPUT_DIR / "D13_branch_B_source_diagnostics.json"
SELECTION_SCOPE_PATH = OUTPUT_DIR / "D13_branch_B_selection_scope.csv"
CELL_AUDIT_PATH = OUTPUT_DIR / "D13_branch_B_cell_audit.csv"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D13_branch_B_structural_markdown.md"
CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D13_branch_B_conversion_integrity.json"
REPRESENTATION_PATH = OUTPUT_DIR / "D13_branch_B_representation.json"
PROMPT_PATH = OUTPUT_DIR / "D13_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D13_branch_B_experiment_metadata_pre.json"

RAW_RESPONSE_PATH = OUTPUT_DIR / "D13_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D13_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D13_branch_B_technical_diagnostics.json"
)
SCOPE_CHECK_PATH = OUTPUT_DIR / "D13_branch_B_scope_check.csv"
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D13_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D13_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected source SHA-256:", EXPECTED_SOURCE_SHA256)
print("Selected scope records:", EXPECTED_RECORD_COUNT)


In [ ]:
# ============================================================
# 1. Upload and verify the exact original D13 workbook
# ============================================================

uploaded = files.upload()

xlsx_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".xlsx")
]

if len(xlsx_paths) != 1:
    raise ValueError(
        "Upload exactly one original D13 XLSX workbook."
    )

SOURCE_PATH = xlsx_paths[0]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def clean_text(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D13 workbook does not match the frozen Stage 1 source identity."
    )

workbook_values = load_workbook(
    SOURCE_PATH,
    data_only=True,
    read_only=False
)

workbook_formulas = load_workbook(
    SOURCE_PATH,
    data_only=False,
    read_only=False
)

sheet_names = workbook_values.sheetnames

worksheet_count_valid = (
    len(sheet_names) == 16
)

worksheet_names_valid = (
    sheet_names == EXPECTED_SHEETS
)

summary_sheet_present = (
    SUMMARY_SHEET in sheet_names
)

selected_sheets_present = all(
    sheet in sheet_names
    for sheet in SELECTED_SHEETS
)

summary_worksheet = workbook_values[
    SUMMARY_SHEET
]

summary_text = " ".join(
    clean_text(cell.value)
    for row in summary_worksheet.iter_rows()
    for cell in row
    if cell.value is not None
)

summary_metadata_checks = {
    "dataset_identifier":
        "lfsa_urgacob" in summary_text,

    "dataset_title":
        "Unemployment rates by country of birth"
        in summary_text,

    "annual_frequency":
        "Annual" in summary_text,

    "percentage_unit":
        "Percentage" in summary_text,

    "age_class":
        "From 15 to 74 years"
        in summary_text
}

summary_metadata_valid = all(
    summary_metadata_checks.values()
)

formula_cell_count = 0
non_empty_cell_count = 0

sheet_shape_rows = []

for sheet_name in sheet_names:

    ws_values = workbook_values[sheet_name]
    ws_formulas = workbook_formulas[sheet_name]

    sheet_non_empty = 0
    sheet_formula_count = 0

    for row in ws_values.iter_rows():
        for cell in row:
            if cell.value is not None:
                non_empty_cell_count += 1
                sheet_non_empty += 1

    for row in ws_formulas.iter_rows():
        for cell in row:
            if (
                isinstance(cell.value, str)
                and cell.value.startswith("=")
            ):
                formula_cell_count += 1
                sheet_formula_count += 1

    sheet_shape_rows.append({
        "Sheet": sheet_name,
        "Rows": ws_values.max_row,
        "Columns": ws_values.max_column,
        "Non-empty Cells": sheet_non_empty,
        "Formula Cells": sheet_formula_count
    })

sheet_shape_df = pd.DataFrame(
    sheet_shape_rows
)

SOURCE_INTEGRITY_VALID = all([
    SOURCE_HASH_MATCH,
    worksheet_count_valid,
    worksheet_names_valid,
    summary_sheet_present,
    selected_sheets_present,
    summary_metadata_valid,
    formula_cell_count == 0
])

SOURCE_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "expected_worksheet_count": 16,
    "observed_worksheet_count": len(sheet_names),
    "worksheet_count_valid": worksheet_count_valid,
    "expected_worksheet_names": EXPECTED_SHEETS,
    "observed_worksheet_names": sheet_names,
    "worksheet_names_valid": worksheet_names_valid,
    "summary_sheet_present": summary_sheet_present,
    "selected_sheets_present": selected_sheets_present,
    "summary_metadata_checks": summary_metadata_checks,
    "summary_metadata_valid": summary_metadata_valid,
    "non_empty_cell_count": non_empty_cell_count,
    "formula_cell_count": formula_cell_count,
    "source_integrity_valid": SOURCE_INTEGRITY_VALID,
    "machine_readable": True,
    "ocr_required": False
}

SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

display(sheet_shape_df)

if not SOURCE_INTEGRITY_VALID:
    raise ValueError(
        "D13 source workbook integrity verification failed."
    )


In [ ]:
# ============================================================
# 2. Derive the fixed Stage 1 extraction scope
# ============================================================

def find_year_columns(worksheet):

    year_columns = {}

    for column_index in range(
        1,
        worksheet.max_column + 1
    ):

        value = worksheet.cell(
            row=11,
            column=column_index
        ).value

        if (
            isinstance(value, int)
            and not isinstance(value, bool)
        ):
            year_columns[value] = column_index

        elif (
            isinstance(value, str)
            and re.fullmatch(
                r"20\d{2}",
                value.strip()
            )
        ):
            year_columns[
                int(value.strip())
            ] = column_index

    return year_columns


def find_geography_rows(worksheet):

    geography_rows = {}

    for row_index in range(
        13,
        worksheet.max_row + 1
    ):

        label = clean_text(
            worksheet.cell(
                row=row_index,
                column=1
            ).value
        )

        if label:
            geography_rows[label] = row_index

    return geography_rows


scope_rows = []

for sheet_name in SELECTED_SHEETS:

    worksheet = workbook_values[
        sheet_name
    ]

    year_columns = find_year_columns(
        worksheet
    )

    geography_rows = find_geography_rows(
        worksheet
    )

    for geography in SELECTED_GEOGRAPHIES:

        if geography not in geography_rows:
            raise ValueError(
                f"{geography!r} was not found in {sheet_name}."
            )

        row_index = geography_rows[
            geography
        ]

        for year in SELECTED_YEARS:

            if year not in year_columns:
                raise ValueError(
                    f"Year {year} was not found in {sheet_name}."
                )

            value_column = year_columns[
                year
            ]

            value_cell = worksheet.cell(
                row=row_index,
                column=value_column
            )

            flag_cell = worksheet.cell(
                row=row_index,
                column=value_column + 1
            )

            scope_rows.append({
                "Sheet": sheet_name,
                "Geography": geography,
                "Year": year,
                "Expected Value Cell": value_cell.coordinate,
                "Expected Flag Cell": flag_cell.coordinate,
                "Expected Source Location":
                    f"{sheet_name}, cell {value_cell.coordinate}"
            })


selection_scope_df = pd.DataFrame(
    scope_rows
)

expected_source_locations = (
    selection_scope_df[
        "Expected Source Location"
    ].tolist()
)

expected_source_location_set = set(
    expected_source_locations
)

scope_structure_valid = all([
    len(selection_scope_df)
    == EXPECTED_RECORD_COUNT,

    len(expected_source_location_set)
    == EXPECTED_RECORD_COUNT
])

SELECTION_SCOPE_PATH.write_text(
    selection_scope_df.to_csv(
        index=False
    ),
    encoding="utf-8"
)

display(selection_scope_df.head(20))

if not scope_structure_valid:
    raise ValueError(
        "D13 fixed extraction-scope structure is invalid."
    )


In [ ]:
# ============================================================
# 3. Convert the COMPLETE 16-sheet workbook to structural Markdown
# ============================================================

def cell_value_type(value):

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "boolean"

    if (
        isinstance(value, int)
        and not isinstance(value, bool)
    ):
        return "integer"

    if isinstance(value, float):
        return "number"

    return "string"


def json_literal(value):

    if (
        isinstance(value, float)
        and value.is_integer()
    ):
        value = int(value)

    return json.dumps(
        value,
        ensure_ascii=False,
        allow_nan=False
    )


cell_records = []
markdown_lines = [
    "# D13 — Eurostat: Unemployment rates by country of birth",
    "",
    "> Complete structural conversion of the original 16-sheet XLSX workbook.",
    "> Every non-empty workbook cell is represented exactly once with its original worksheet and cell coordinate.",
    "> The fixed extraction scope is defined only by the extraction prompt; the representation itself is not filtered.",
    ""
]

for sheet_name in sheet_names:

    worksheet = workbook_values[
        sheet_name
    ]

    markdown_lines.extend([
        f"## Worksheet: {sheet_name}",
        ""
    ])

    for row_index in range(
        1,
        worksheet.max_row + 1
    ):

        row_records = []

        for column_index in range(
            1,
            worksheet.max_column + 1
        ):

            cell = worksheet.cell(
                row=row_index,
                column=column_index
            )

            if cell.value is None:
                continue

            value_type = cell_value_type(
                cell.value
            )

            encoded_value = json_literal(
                cell.value
            )

            row_records.append({
                "Sheet": sheet_name,
                "Row": row_index,
                "Cell": cell.coordinate,
                "Type": value_type,
                "Encoded Value": encoded_value,
                "Original Value": cell.value
            })

        if not row_records:
            continue

        markdown_lines.extend([
            f"### Row {row_index}",
            ""
        ])

        for record in row_records:

            markdown_lines.append(
                f"- {record['Cell']} "
                f"[{record['Type']}]: "
                f"{record['Encoded Value']}"
            )

            cell_records.append(
                record
            )

        markdown_lines.append("")


STRUCTURAL_MARKDOWN_TEXT = (
    "\n".join(markdown_lines)
    .rstrip()
    + "\n"
)

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN_TEXT,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)

cell_audit_df = pd.DataFrame(
    cell_records
)

cell_audit_df.to_csv(
    CELL_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Structural Markdown:",
    STRUCTURAL_MARKDOWN_PATH
)

print(
    "Representation SHA-256:",
    STRUCTURAL_MARKDOWN_SHA256
)

print(
    "Represented non-empty cells:",
    len(cell_records)
)


In [ ]:
# ============================================================
# 4. Conversion-integrity verification
# ============================================================

worksheet_header_pattern = re.compile(
    r"^## Worksheet:\s*(.+?)\s*$"
)

cell_line_pattern = re.compile(
    r"^-\s+([A-Z]+\d+)\s+\[([a-z]+)\]:\s+(.+)$"
)

parsed_cells = []

current_sheet = None

for line in STRUCTURAL_MARKDOWN_TEXT.splitlines():

    worksheet_match = (
        worksheet_header_pattern.fullmatch(
            line.strip()
        )
    )

    if worksheet_match:

        current_sheet = (
            worksheet_match.group(1)
        )

        continue

    cell_match = cell_line_pattern.fullmatch(
        line.strip()
    )

    if (
        cell_match
        and current_sheet is not None
    ):

        coordinate = cell_match.group(1)
        value_type = cell_match.group(2)
        encoded_value = cell_match.group(3)

        try:
            decoded_value = json.loads(
                encoded_value
            )
        except json.JSONDecodeError:
            raise ValueError(
                f"Could not decode converted cell {current_sheet}!{coordinate}."
            )

        parsed_cells.append({
            "Sheet": current_sheet,
            "Cell": coordinate,
            "Type": value_type,
            "Encoded Value": encoded_value,
            "Decoded Value": decoded_value
        })


parsed_cell_df = pd.DataFrame(
    parsed_cells
)

source_signature = [
    (
        record["Sheet"],
        record["Cell"],
        record["Type"],
        record["Encoded Value"]
    )
    for record in cell_records
]

converted_signature = [
    (
        record["Sheet"],
        record["Cell"],
        record["Type"],
        record["Encoded Value"]
    )
    for record in parsed_cells
]


conversion_cell_count_valid = (
    len(parsed_cells)
    == non_empty_cell_count
)

conversion_sheet_headers_valid = all(
    f"## Worksheet: {sheet_name}"
    in STRUCTURAL_MARKDOWN_TEXT
    for sheet_name in sheet_names
)

conversion_sheet_order_valid = (
    [
        match.group(1)
        for line in STRUCTURAL_MARKDOWN_TEXT.splitlines()
        if (
            match := worksheet_header_pattern.fullmatch(
                line.strip()
            )
        )
    ]
    == sheet_names
)

conversion_cells_identical = (
    converted_signature
    == source_signature
)

converted_cell_keys = {
    (
        record["Sheet"],
        record["Cell"]
    )
    for record in parsed_cells
}

source_cell_keys = {
    (
        record["Sheet"],
        record["Cell"]
    )
    for record in cell_records
}

all_source_cells_preserved = (
    converted_cell_keys
    == source_cell_keys
)

selected_scope_cells_present = all(
    (
        row["Sheet"],
        row["Expected Value Cell"]
    )
    in converted_cell_keys
    for _, row
    in selection_scope_df.iterrows()
)

selected_flag_cells_present = all(
    (
        row["Sheet"],
        row["Expected Flag Cell"]
    )
    in converted_cell_keys
    or
    workbook_values[
        row["Sheet"]
    ][
        row["Expected Flag Cell"]
    ].value
    is None
    for _, row
    in selection_scope_df.iterrows()
)

CONVERSION_INTEGRITY_PASSED = all([
    SOURCE_INTEGRITY_VALID,
    conversion_cell_count_valid,
    conversion_sheet_headers_valid,
    conversion_sheet_order_valid,
    conversion_cells_identical,
    all_source_cells_preserved,
    selected_scope_cells_present,
    selected_flag_cells_present
])

CONVERSION_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "source_worksheet_count":
        len(sheet_names),

    "converted_worksheet_count":
        len(sheet_names),

    "source_non_empty_cell_count":
        non_empty_cell_count,

    "converted_cell_count":
        len(parsed_cells),

    "conversion_cell_count_valid":
        conversion_cell_count_valid,

    "conversion_sheet_headers_valid":
        conversion_sheet_headers_valid,

    "conversion_sheet_order_valid":
        conversion_sheet_order_valid,

    "conversion_cells_identical":
        conversion_cells_identical,

    "all_source_cells_preserved":
        all_source_cells_preserved,

    "selected_scope_value_cells_present":
        selected_scope_cells_present,

    "selected_scope_flag_cells_present_when_nonempty":
        selected_flag_cells_present,

    "conversion_method":
        CONVERSION_METHOD,

    "complete_source_workbook_retained":
        True,

    "worksheet_filtering_applied":
        False,

    "row_filtering_applied":
        False,

    "column_filtering_applied":
        False,

    "selected_scope_filtering_applied":
        False,

    "cell_reordering_applied":
        False,

    "aggregation_applied":
        False,

    "interpolation_applied":
        False,

    "ocr_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "value_conversion_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "normalisation_applied":
        False,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED
}

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)

if not CONVERSION_INTEGRITY_PASSED:
    raise ValueError(
        "D13 Branch B structural conversion failed integrity checks."
    )


In [ ]:
# ============================================================
# 5. Preserve representation metadata
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Complete coordinate-aware structural Markdown workbook",

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "branch_name":
        BRANCH_NAME,

    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "source_worksheet_count":
        len(sheet_names),

    "converted_worksheet_count":
        len(sheet_names),

    "source_non_empty_cell_count":
        non_empty_cell_count,

    "converted_non_empty_cell_count":
        len(parsed_cells),

    "structural_conversion_applied":
        True,

    "cell_coordinates_preserved":
        True,

    "cell_source_types_preserved":
        True,

    "complete_source_workbook_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "worksheet_filtering_applied":
        False,

    "row_filtering_applied":
        False,

    "column_filtering_applied":
        False,

    "selected_scope_filtering_applied":
        False,

    "ocr_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "value_conversion_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED
}

REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 6. Create controlled Branch B extraction prompt
# ============================================================

sheet_lines = "\n".join(
    f"- {sheet_name}"
    for sheet_name in SELECTED_SHEETS
)

geography_lines = "\n".join(
    f"- {geography}"
    for geography in SELECTED_GEOGRAPHIES
)

year_lines = "\n".join(
    f"- {year}"
    for year in SELECTED_YEARS
)


BRANCH_B_PROMPT = f"""You are an information extraction assistant.

Extract the predefined unemployment-rate observations represented in
the attached structural Markdown representation of the workbook:

"Unemployment rates by country of birth"

Treat the attached structural Markdown representation as the only
source of information.

The representation contains the complete original workbook and
preserves worksheet names, physical cell coordinates and source values.


Selected worksheets:

{sheet_lines}


Selected geographies:

{geography_lines}


Selected reporting years:

{year_lines}


For every represented combination of selected worksheet, selected
geography and selected reporting year, return one record.

For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location


Use these fixed semantic values:

Category:
Labour market time series

Topic:
Unemployment rate by country of birth

Unit:
percent


Worksheet-level dimensions:

For each selected worksheet, read the represented worksheet-level
dimensions directly from the structural Markdown:

- Sex
- Age Class
- Country/Region of Birth

Do not infer these dimensions from the worksheet number or from another
worksheet.


Description:

Construct Description in exactly this order:

Geography: <geography>; Sex: <sex>; Age class: <age class>;
Country/region of birth: <birth category>

Use the dimension wording represented in the corresponding worksheet.

When the statistical-flag cell directly adjacent to the selected value
cell is non-empty, append:

; Statistical flag: <flag>

Do not append a Statistical flag segment when the adjacent flag cell
is blank.

Preserve the represented statistical flag exactly.
Do not infer, rewrite, expand or interpret statistical flags.


Value:

- Extract the numerical unemployment-rate value represented for the
  selected geography and selected reporting year.
- Use the value cell, not the adjacent statistical-flag cell.
- Return the value as a JSON number.
- Do not calculate, aggregate, interpolate, round, convert or correct
  the represented value.


Reporting Period:

- Use the selected reporting year as a four-digit string.
- Example:
  "2024"


Source Location:

- Identify the exact workbook cell coordinate represented for the
  numerical value.
- Use this form:
  "Sheet N, cell A1"
- Use the value cell, not the adjacent statistical-flag cell.
- Determine the cell directly from the structural Markdown.


Extraction rules:

- Use only the selected worksheets.
- Use only the selected geographies.
- Use only the selected reporting years.
- Return one observation for every represented combination within this
  predefined scope.
- Do not omit a required combination.
- Do not return observations outside the predefined scope.
- Read Sex, Age Class and Country/Region of Birth directly from each
  selected worksheet.
- Preserve exact geography and dimension labels.
- Preserve an adjacent statistical flag only when explicitly represented.
- Do not calculate missing observations.
- Do not aggregate values.
- Do not interpolate values.
- Do not convert percentages into another scale.
- Do not infer missing statistical flags.
- Do not use external knowledge.
- Do not use observations from unselected worksheets.
- Do not duplicate records.
- Ignore structural Markdown headings and source-type labels except as
  aids for locating worksheet, row and cell structure.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.


Expected JSON structure:

{{
  "document_id": "D13",
  "branch": "B",
  "records": [
    {{
      "Category": "Labour market time series",
      "Topic": "Unemployment rate by country of birth",
      "Description": null,
      "Value": null,
      "Unit": "percent",
      "Reporting Period": null,
      "Source Location": null
    }}
  ]
}}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print(BRANCH_B_PROMPT)


In [ ]:
# ============================================================
# 7. Pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_INTEGRITY_VALID,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "complete_source_workbook_retained":
        True,

    "worksheet_filtering_applied_to_model_input":
        False,

    "row_filtering_applied_to_model_input":
        False,

    "column_filtering_applied_to_model_input":
        False,

    "selected_scope_filtering_applied_to_model_input":
        False,

    "cell_coordinates_preserved":
        True,

    "ocr_applied_for_model_input":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "value_conversion_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "selected_sheets":
        SELECTED_SHEETS,

    "selected_geographies":
        SELECTED_GEOGRAPHIES,

    "selected_years":
        SELECTED_YEARS,

    "fixed_extraction_task": {
        "expected_fields":
            EXPECTED_FIELDS
    },

    "post_extraction_reference_diagnostics": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_year_counts":
            EXPECTED_YEAR_COUNTS
    },

    "reference_values_disclosed_to_model":
        False,

    "reference_record_count_explicitly_disclosed_to_model":
        False,

    "reference_year_counts_explicitly_disclosed_to_model":
        False,

    "worksheet_dimension_answers_disclosed_to_model":
        False,

    "statistical_flag_answers_disclosed_to_model":
        False,

    "source_cell_reference_answers_disclosed_to_model":
        False,

    "selected_scope_disclosed_to_model":
        True,

    "source_diagnostics_file":
        SOURCE_DIAGNOSTICS_PATH.name,

    "selection_scope_file":
        SELECTION_SCOPE_PATH.name,

    "cell_audit_file":
        CELL_AUDIT_PATH.name,

    "conversion_integrity_file":
        CONVERSION_INTEGRITY_PATH.name,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        "JSON object with document_id, branch and records",

    "execution_environment":
        "Independent ChatGPT conversation",

    "content_validation_performed":
        False,
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 8. Download Branch B model-input artefacts
# ============================================================

for path in [
    SOURCE_DIAGNOSTICS_PATH,
    SELECTION_SCOPE_PATH,
    CELL_AUDIT_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D13_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D13_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original XLSX, Stage 1 reference dataset, "
    "Branch A outputs, or Validation A outputs.\n"
    "5. Save the first complete model response exactly as returned as TXT.\n"
    "6. Do not correct, repair, reorder, deduplicate or regenerate it."
)


In [ ]:
# ============================================================
# 9. Upload and preserve the untouched model response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete D13 Branch B response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]

RAW_RESPONSE_TEXT = (
    UPLOADED_RAW_RESPONSE_PATH
    .read_text(
        encoding="utf-8"
    )
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError(
        "The uploaded D13 Branch B response is empty."
    )

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print(
    "Raw response preserved:",
    RAW_RESPONSE_PATH.name
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


In [ ]:
# ============================================================
# 10. Parse without repairing the response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:

    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

    valid_json = True

except json.JSONDecodeError as error:

    json_parsing_error = str(error)


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

parsed_extraction_created = False
parsed_extraction_sha256 = None


print("Valid JSON:", valid_json)
print("JSON parsing error:", json_parsing_error)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Records evaluable:", records_evaluable)
print("Observed record count:", observed_record_count)


In [ ]:
# ============================================================
# 11. Schema, field-type and D13 scope diagnostics
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(record, dict):

            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue

        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:

            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })

        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "string or null"
                })

        value = record.get(
            "Value"
        )

        if (
            value is not None
            and (
                isinstance(
                    value,
                    bool
                )
                or not isinstance(
                    value,
                    (int, float)
                )
            )
        ):

            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(
                        value
                    ).__name__,

                "expected_type":
                    "number or null"
            })

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )

            if (
                value is None
                or (
                    isinstance(value, str)
                    and not value.strip()
                )
            ):

                missing_mandatory_values.append({
                    "record_index":
                        record_index,

                    "field":
                        field
                })


record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)


if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )

    observed_year_counts = dict(
        Counter(
            str(
                record.get(
                    "Reporting Period"
                )
            ).strip()
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    year_counts_valid = (
        observed_year_counts
        == EXPECTED_YEAR_COUNTS
    )

    category_constant_valid = all(
        record.get("Category")
        == EXPECTED_CATEGORY
        for record in extracted_records
        if isinstance(record, dict)
    )

    topic_constant_valid = all(
        record.get("Topic")
        == EXPECTED_TOPIC
        for record in extracted_records
        if isinstance(record, dict)
    )

    unit_constant_valid = all(
        record.get("Unit")
        == EXPECTED_UNIT
        for record in extracted_records
        if isinstance(record, dict)
    )

    constant_fields_valid = all([
        category_constant_valid,
        topic_constant_valid,
        unit_constant_valid
    ])

    DESCRIPTION_LABELS = [
        "Geography:",
        "Sex:",
        "Age class:",
        "Country/region of birth:"
    ]

    description_dimension_labels_valid = all(
        isinstance(
            record.get(
                "Description"
            ),
            str
        )
        and all(
            label
            in record.get(
                "Description",
                ""
            )
            for label in DESCRIPTION_LABELS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    observed_flag_segment_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get(
                    "Description"
                ),
                str
            )
            and "Statistical flag:"
            in record["Description"]
        )
    )

    source_location_pattern = re.compile(
        r"^Sheet [1-5], cell [A-Z]+\d+$"
    )

    observed_source_locations = [
        record.get(
            "Source Location"
        )
        for record in extracted_records
        if isinstance(record, dict)
    ]

    source_location_format_valid = all(
        isinstance(location, str)
        and source_location_pattern.fullmatch(
            location.strip()
        )
        is not None
        for location in observed_source_locations
    )

    observed_source_location_set = set(
        observed_source_locations
    )

    expected_source_locations_complete = (
        observed_source_location_set
        == expected_source_location_set
    )

    extracted_source_locations_unique = (
        len(observed_source_locations)
        == len(observed_source_location_set)
    )

    unexpected_source_locations = sorted(
        observed_source_location_set
        - expected_source_location_set
    )

    missing_source_locations = sorted(
        expected_source_location_set
        - observed_source_location_set
    )

    fixed_scope_valid = all([
        expected_source_locations_complete,
        extracted_source_locations_unique
    ])

    duplicate_complete_record_signature_count = sum(
        1
        for count
        in Counter(
            tuple(
                json.dumps(
                    record.get(field),
                    ensure_ascii=False,
                    sort_keys=True
                )
                for field
                in EXPECTED_FIELDS
            )
            for record
            in extracted_records
            if isinstance(record, dict)
        ).values()
        if count > 1
    )

else:

    record_count_valid = None
    observed_category_counts = None
    category_counts_valid = None
    categories_valid = None
    observed_year_counts = None
    year_counts_valid = None
    category_constant_valid = None
    topic_constant_valid = None
    unit_constant_valid = None
    constant_fields_valid = None
    description_dimension_labels_valid = None
    observed_flag_segment_count = None
    observed_source_locations = None
    source_location_format_valid = None
    expected_source_locations_complete = None
    extracted_source_locations_unique = None
    unexpected_source_locations = None
    missing_source_locations = None
    fixed_scope_valid = None
    duplicate_complete_record_signature_count = None


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match_reference":
        category_counts_valid,

    "expected_year_counts":
        EXPECTED_YEAR_COUNTS,

    "observed_year_counts":
        observed_year_counts,

    "year_counts_match_reference":
        year_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(missing_mandatory_values)
            if records_evaluable
            else None
        ),

    "constant_fields_valid":
        constant_fields_valid,

    "description_dimension_labels_valid":
        description_dimension_labels_valid,

    "observed_flag_segment_count":
        observed_flag_segment_count,

    "source_location_format_valid":
        source_location_format_valid,

    "expected_source_locations_complete":
        expected_source_locations_complete,

    "source_locations_unique":
        extracted_source_locations_unique,

    "unexpected_source_locations":
        unexpected_source_locations,

    "missing_source_locations":
        missing_source_locations,

    "fixed_scope_valid":
        fixed_scope_valid,

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_signature_count
}

scope_check_rows = []

if records_evaluable:

    for expected_location in sorted(
        expected_source_location_set
    ):

        occurrence_count = (
            observed_source_locations.count(
                expected_location
            )
        )

        scope_check_rows.append({
            "Expected Source Location":
                expected_location,

            "Observed Count":
                occurrence_count,

            "Valid":
                occurrence_count == 1
        })

scope_check_df = pd.DataFrame(
    scope_check_rows
)

scope_check_df.to_csv(
    SCOPE_CHECK_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 12. Determination of technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "missing_mandatory_values":
        missing_mandatory_values
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(scope_complete)
        if scope_complete is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 13. Preserve parsed extraction if structurally evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),

        "branch":
            parsed_response.get("branch"),

        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True

    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:
    print(
        "No parsed extraction created because "
        "the output is not structurally evaluable."
    )

In [ ]:
# ============================================================
# 14. Final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "observed_year_counts":
        observed_year_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "content_validation_performed":
        False,

    "scope_check_file":
        SCOPE_CHECK_PATH.name,

}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_INTEGRITY_VALID,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED,

    "structural_conversion_applied":
        True,

    "complete_source_workbook_retained":
        True,

    "source_worksheet_count":
        len(sheet_names),

    "represented_non_empty_cell_count":
        len(parsed_cells),

    "selected_sheet_count":
        len(SELECTED_SHEETS),

    "selected_geography_count":
        len(SELECTED_GEOGRAPHIES),

    "selected_year_count":
        len(SELECTED_YEARS),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "expected_year_counts":
        EXPECTED_YEAR_COUNTS,

    "observed_year_counts":
        observed_year_counts,

    "year_counts_match":
        year_counts_valid,

    "valid_json":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "scope_complete":
        scope_complete,

    "constant_fields_valid":
        constant_fields_valid,

    "description_dimension_labels_valid":
        description_dimension_labels_valid,

    "observed_flag_segment_count":
        observed_flag_segment_count,

    "source_location_format_valid":
        source_location_format_valid,

    "expected_source_locations_complete":
        expected_source_locations_complete,

    "source_locations_unique":
        extracted_source_locations_unique,

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_signature_count,

    "parsed_extraction_created":
        bool(structurally_evaluable),

    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D13."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 15. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    SOURCE_DIAGNOSTICS_PATH,
    SELECTION_SCOPE_PATH,
    CELL_AUDIT_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SCOPE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(
        PARSED_EXTRACTION_PATH
    )

print(
    "Generated D13 Branch B files:"
)

for path in GENERATED_OUTPUTS:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )

for path in GENERATED_OUTPUTS:

    files.download(
        path
    )
